# 🤖 AI-102 Exercise 03 — Create a Generative AI Chat App

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Copilotclaw/trainingdemo/blob/main/kurs/ai102-03-foundry-sdk.ipynb)

Based on: [03-foundry-sdk.md](https://github.com/MicrosoftLearning/mslearn-ai-studio/blob/main/Instructions/Exercises/03-foundry-sdk.md)

In this notebook you will use the **OpenAI SDK** and the **Responses API** to build a chat app that connects to a model deployed in **Microsoft Foundry**.

Estimated time: **45 minutes**

---

## 📋 Portal steps (do these first, outside this notebook)

### 1. Create a Microsoft Foundry project

1. Open the [Microsoft Foundry portal](https://ai.azure.com) and sign in.
2. Enable the **New Foundry** option in the toolbar (if not already on).
3. Create a new project with a unique name, expanding **Advanced options** to set:
   - **Foundry resource**: accept the default
   - **Subscription**: your Azure subscription
   - **Resource group**: create or select one
   - **Region**: any **AI Foundry recommended** region
4. Click **Create** and wait.

### 2. Deploy a model

1. On the project home page → **Start building** → **Browse models**.
2. Search for `gpt-4.1`, open the model card, and deploy with default settings.
3. (Optional) Test it in the playground.

### 3. Get your endpoint and key

1. Navigate to the **Home** page of your project.
2. Copy the **Project API key** and **Azure OpenAI Endpoint** — you'll need them below.


## 🔑 Step 1 — Credentials

Choose **one** of the three options below and run only that cell.

| Option | How | Best for |
|--------|-----|----------|
| **A** | Paste values directly | Classroom / quick demo |
| **B** | Colab Secrets | Personal reuse (keys never saved in .ipynb) |
| **C** | Upload `.env` file | Same setup as the original VS Code lab |


In [ ]:
# ── OPTION A: paste directly ─────────────────────────────────────────────────
azure_openai_endpoint = "https://<your-resource>.openai.azure.com/"  # paste here
api_key               = "<your-api-key>"                             # paste here
model_deployment      = "gpt-4.1"                                    # change if you renamed it
print("✅ Credentials set (Option A)")

In [ ]:
# ── OPTION B: Colab Secrets ──────────────────────────────────────────────────
# Add three secrets in the 🔑 Colab sidebar:
#   AZURE_OPENAI_ENDPOINT  /  API_KEY  /  MODEL_DEPLOYMENT
from google.colab import userdata
azure_openai_endpoint = userdata.get("AZURE_OPENAI_ENDPOINT")
api_key               = userdata.get("API_KEY")
model_deployment      = userdata.get("MODEL_DEPLOYMENT") or "gpt-4.1"
print("✅ Credentials loaded from Colab Secrets (Option B)")

In [ ]:
# ── OPTION C: .env file ───────────────────────────────────────────────────────
# Upload your .env file (same as the original lab) using the Files pane (📁)
# Expected contents:
#   AZURE_OPENAI_ENDPOINT=https://...
#   API_KEY=...
#   MODEL_DEPLOYMENT=gpt-4.1
import os
from dotenv import load_dotenv
load_dotenv()
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_key               = os.getenv("API_KEY")
model_deployment      = os.getenv("MODEL_DEPLOYMENT") or "gpt-4.1"
print("✅ Credentials loaded from .env (Option C)")

## 📦 Step 2 — Install dependencies


In [ ]:
%pip install -q openai python-dotenv
print("✅ Dependencies installed")

## 💬 Part 1 — ChatCompletions API

The **ChatCompletions** API is the well-established way to chat with LLMs.  
It uses a list of `messages` with `system` and `user` roles.


In [ ]:
from openai import OpenAI

openai_client = OpenAI(
    base_url=azure_openai_endpoint,
    api_key=api_key
)

def chat_completions(user_input: str) -> str:
    completion = openai_client.chat.completions.create(
        model=model_deployment,
        messages=[
            {
                "role": "system",
                "content": "You are a helpful AI assistant that answers questions and provides information."
            },
            {
                "role": "user",
                "content": user_input
            }
        ]
    )
    return completion.choices[0].message.content

# Test it
response = chat_completions("Tell me about the ELIZA chatbot.")
print(response)

## 💬 Part 2 — Responses API (simpler syntax)

The newer **Responses API** has a cleaner interface:
- `instructions` replaces the system message
- `input` replaces the user message list

> ⚠️ Note: the Responses API does **not** carry conversation context automatically — each call is independent.


In [ ]:
def responses_api(user_input: str) -> str:
    response = openai_client.responses.create(
        model=model_deployment,
        instructions="You are a helpful AI assistant that answers questions and provides information.",
        input=user_input
    )
    return response.output_text

# Test — same question
print(responses_api("Tell me about the ELIZA chatbot."))

In [ ]:
# Try a follow-up — notice context is LOST ("it" won't be understood)
print(responses_api("How does it compare to modern LLMs?"))

## 🧵 Part 3 — Conversation tracking with `previous_response_id`

Pass `previous_response_id` to carry context across turns.  
The model uses the previous response to understand follow-up questions.


In [ ]:
last_response_id = None

def chat_with_context(user_input: str) -> str:
    global last_response_id
    response = openai_client.responses.create(
        model=model_deployment,
        instructions="You are a helpful AI assistant that answers questions and provides information.",
        input=user_input,
        previous_response_id=last_response_id
    )
    last_response_id = response.id
    return response.output_text

# Turn 1
print("USER: Tell me about the ELIZA chatbot.")
print("ASSISTANT:", chat_with_context("Tell me about the ELIZA chatbot."))

In [ ]:
# Turn 2 — now "it" correctly refers to ELIZA
print("USER: How does it compare to modern LLMs?")
print("ASSISTANT:", chat_with_context("How does it compare to modern LLMs?"))

## ⚡ Part 4 — Streaming responses

For long answers, **streaming** shows output token-by-token as it arrives — no more waiting for the full response.


In [ ]:
last_response_id = None  # reset conversation

def chat_streaming(user_input: str):
    global last_response_id
    stream = openai_client.responses.create(
        model=model_deployment,
        instructions="You are a helpful AI assistant that answers questions and provides information.",
        input=user_input,
        previous_response_id=last_response_id,
        stream=True
    )
    for event in stream:
        if event.type == "response.output_text.delta":
            print(event.delta, end="", flush=True)
        elif event.type == "response.completed":
            last_response_id = event.response.id
    print()  # newline at end

print("USER: Tell me about the ELIZA chatbot.")
print("ASSISTANT: ", end="")
chat_streaming("Tell me about the ELIZA chatbot.")

In [ ]:
print("USER: How does it compare to modern LLMs?")
print("ASSISTANT: ", end="")
chat_streaming("How does it compare to modern LLMs?")

## 🎮 Interactive Chat (bonus)

Run the cell below for a full interactive streaming chat loop in the notebook. Type `quit` to exit.


In [ ]:
last_response_id = None  # fresh conversation

print("🤖 Chat started — type 'quit' to exit\n")
while True:
    user_input = input("You: ").strip()
    if not user_input:
        continue
    if user_input.lower() == "quit":
        print("👋 Bye!")
        break
    print("Assistant: ", end="")
    chat_streaming(user_input)

## ✅ Summary

In this exercise you:

| Step | What you did |
|------|-------------|
| Part 1 | Used the **ChatCompletions** API (classic approach) |
| Part 2 | Switched to the **Responses** API (cleaner syntax) |
| Part 3 | Added **conversation context** via `previous_response_id` |
| Part 4 | Implemented **streaming** for responsive output |

## 🧹 Clean up

When done, delete the Azure resources to avoid unnecessary costs:

1. Open the [Azure portal](https://portal.azure.com) and find the resource group you created.
2. Select **Delete resource group**, enter the name, and confirm.
